# Optical Spectrum Classification & Analysis Pipeline
-------
## Overview
This notebook implements an automated pipeline for the classification and analysis of galaxy optical spectra. It distinguishes between various galaxy types—**Star-Forming (SF), Active Galactic Nuclei (AGN: Broad and narrow), LINER, Composite, and Passive**—using machine learning models trained on spectral features.

This notebook is tailored for applying the diagnostic to large catalogs that have already measuremnts for the EWs 

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import EW_classifier as ewc
import joblib

In [ ]:
clf = joblib.load('../Models/Auto_OptSepcClassifier_SVM.sav')
scaler_sv = joblib.load('../Models/Auto_OptSepcClassifier_scaler.sav')

In [ ]:
scaler_AGN_Type = joblib.load('../Models/Auto_OptSepcClassifier_scaler_AGN_type.sav')
clf_AGN_Type = joblib.load('../Models/Auto_OptSepcClassifier_SVM_AGN_type.sav')

In [ ]:
path_to_AGN_spec = '../Example data/SDSS_example_spectrum_broad_line_AGN.csv'
path_to_SFG_spec = '../Example data/SDSS_example_spectrum_SFG.csv'

df_csv = pd.read_csv(f'{path_to_SFG_spec}')

flux = df_csv['flux']
wavelength = df_csv['wavelength']
var = df_csv['var']

In [ ]:
columns = ['EW_NII_Ha', 'EW_OIII', 'EW_Hb', 'EW_NII_Ha_unc', 'EW_OIII_unc', 'EW_Hb_unc']

data = [
    [-97.55, -9.28, -10.18, 1.37, 0.43, 0.38],
    [-96.10, -9.05, -10.50, 1.40, 0.41, 0.39]
    ]

df_example = pd.DataFrame(data, columns=columns)

In [ ]:
rows = []
for index, row in df_example.iterrows():
    label, means, stds = ewc.rf_classify_mc_EWs(
        row[['EW_NII_Ha', 'EW_OIII', 'EW_Hb']], 
        row[['EW_NII_Ha_unc', 'EW_OIII_unc', 'EW_Hb_unc']], 
        clf=clf, 
        scaler_sv=scaler_sv,
        n_mc=1000,   
        plot=False   
    )
    res = {**{'label': label}, **means.add_suffix('_mean').to_dict(), **stds.add_suffix('_std').to_dict()}
    rows.append(res)

df_results = pd.DataFrame(rows, index=df_example.index)
df_final = pd.concat([df_example, df_results], axis=1)
df_final

In [ ]:
df_final.to_csv('path-to-save/classification_result.csv', index=False)

In [ ]:
# EOF